# MuonClip Angular WeightWatcher — multi-seed initial / best / final

This notebook analyzes **all completed MuonClip seeds** under the configured results root. Each seed must contain the actual saved `checkpoint_initial.pt`, `checkpoint_best.pt`, and `checkpoint_final.pt`.

For every seed, the shared angular engine evaluates $W_0\to W_{best}$, $W_0\to W_{final}$, and $W_{best}\to W_{final}$ in the gauge-invariant `tilt` and `twist` sectors, with a matched Haar/Stiefel random-angular null.

The scientific replicate is the **training seed**. Random-null realizations are never counted as seeds. Cross-seed error bars are **95% Student-t confidence intervals** around the seed mean, and every individual seed value is plotted. For every core metric the notebook generates a full-range plot and an additional **zoomed-y** plot. The zoom changes only display limits; it never changes any fitted data.


## Power-law and random-null contract

For a bounded angular eigenvalue $\lambda$, the continuous projective coordinate is $x=(\lambda/u)/(1-\lambda/u)$. Upper-endpoint atoms are counted separately instead of being clipped into artificial giant tail values.

Every remaining positive continuous value is passed to `powerlaw.Fit(values, discrete=False, verbose=False)` with **no `xmin` and no `xmax` supplied**. The package selects $x_{min}$ by its MLE/KS procedure and retains the largest observed continuous values.

The per-seed analysis records alpha, package-selected $x_{min}$, KS $D$, tail count, tail length, endpoint atoms, and Monte-Carlo comparisons with the matched random-angular null. The multi-seed layer aggregates seed-level statistics only.


## Papermill and environment usage

Set `RESULTS_ROOT` to the common campaign results directory and normally leave `RUN_DIR` unset. `ANGULAR_SEEDS` may be blank to auto-discover every completed seed, or a comma/space separated list such as `1337,2027,31415,271828`.

```bash
export RG_OPTIMIZERS_ROOT=/tmp/rg_optimizers
export RUNROOT=/tmp/<campaign-run-root>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
unset RUN_DIR INITIAL_CHECKPOINT_PATH BEST_CHECKPOINT_PATH FINAL_CHECKPOINT_PATH
export ANGULAR_SEEDS=""
export ANGULAR_N_NULL=100
export ANGULAR_MIN_TAIL=20
export ANGULAR_ENDPOINT_TOL=1e-10
export ANGULAR_SHOW_PLOTS=0

papermill baseline/nanogpt_one_head/notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb /tmp/angular_multiseed.out.ipynb
```

Increase `ANGULAR_N_NULL` after the first successful run if a stronger random-null ensemble is desired. Papermill `-p` overrides work because configuration is built after the tagged parameters cell.


In [ ]:
import os
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", os.environ.get("OPTIMIZER_NAME", "muon_clip"))
RESULTS_ROOT = os.environ.get("RESULTS_ROOT", "")
RUNROOT = os.environ.get("RUNROOT", "")
RUN_DIR = os.environ.get("RUN_DIR", "")
ANGULAR_SEEDS = os.environ.get("ANGULAR_SEEDS", "")
ANGULAR_OUTPUT_DIR = os.environ.get("ANGULAR_OUTPUT_DIR", "")
ANGULAR_N_NULL = int(os.environ.get("ANGULAR_N_NULL", "100"))
ANGULAR_N_ENTRY_NULL = int(os.environ.get("ANGULAR_N_ENTRY_NULL", "24"))
ANGULAR_MIN_TAIL = int(os.environ.get("ANGULAR_MIN_TAIL", "20"))
ANGULAR_NULL_SEED = int(os.environ.get("ANGULAR_NULL_SEED", "91337"))
ANGULAR_ENDPOINT_TOL = float(os.environ.get("ANGULAR_ENDPOINT_TOL", "1e-10"))
ANGULAR_SHOW_PLOTS = os.environ.get("ANGULAR_SHOW_PLOTS", "0")


In [ ]:
from pathlib import Path
import sys

def _as_bool(value):
    if isinstance(value, bool): return value
    return str(value).strip().lower() not in {"0", "false", "no", "off"}

def _none_if_blank(value):
    text = str(value).strip() if value is not None else ""
    return text or None

def find_experiment_root():
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file(): return candidate
    raise FileNotFoundError("Set RG_OPTIMIZERS_ROOT or launch from rg_optimizers")

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
os.environ["ANGULAR_ENDPOINT_TOL"] = str(ANGULAR_ENDPOINT_TOL)
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_multiseed import run_multiseed_analysis
CONFIG = AnalysisConfig(optimizer=str(TARGET_OPTIMIZER).lower(), results_root=_none_if_blank(RESULTS_ROOT), runroot=_none_if_blank(RUNROOT), run_dir=_none_if_blank(RUN_DIR), output_dir=_none_if_blank(ANGULAR_OUTPUT_DIR), angular_nulls=int(ANGULAR_N_NULL), entry_nulls=int(ANGULAR_N_ENTRY_NULL), min_tail=int(ANGULAR_MIN_TAIL), null_seed=int(ANGULAR_NULL_SEED), show_plots=_as_bool(ANGULAR_SHOW_PLOTS))
print("RESULTS_ROOT =", RESULTS_ROOT or "<derived>")
print("RUN_DIR =", RUN_DIR or "<unset; preferred for multi-seed>")
print("ANGULAR_SEEDS =", ANGULAR_SEEDS or "<auto-discover all completed seeds>")


In [ ]:
from IPython.display import display
SEED_RESULTS, CROSS_SEED, MANIFEST = run_multiseed_analysis(CONFIG, seed_spec=str(ANGULAR_SEEDS))
print("Seeds analyzed:", MANIFEST["seeds"])
print("n_seeds:", MANIFEST["n_seeds"])
print("Error bars:", MANIFEST["error_bar_contract"])
print("Output directory:", MANIFEST["output_dir"])
display(CROSS_SEED)


In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path
for metric in ("alpha", "tail_decades", "D", "xmin"):
    display(Markdown(f"## Cross-seed {metric.replace('_', ' ')}"))
    for suffix in ("full_y", "zoom_y"):
        for path in [Path(p) for p in MANIFEST["plots"] if f"cross_seed_{metric}_" in Path(p).name and suffix in Path(p).name]:
            display(Markdown(f"### `{path.name}`"))
            display(Image(filename=str(path), width=1100))


## Interpretation

Dots are individual training seeds. Trained and random-null summaries show mean $\pm$ 95% Student-t confidence intervals across seeds. Full-y plots preserve the complete scale; zoom-y plots magnify reproducible differences without changing any fit. For alpha, the $\alpha=2$ reference remains visible.

Read alpha together with tail length, package-selected $x_{min}$, and KS $D$. A power-law-looking fit that is not reproducible across seeds or cannot be distinguished from the random-angular baseline is not evidence for angular RG flow.
